# NB02 — Stratification & Chunking Strategy

This notebook transforms the discoveries from NB01 into an operational framework.

**Decision thread:**
1. Official corpus stratification (16 strata)
2. Decision matrix: index type, role, priority per stratum
3. Comparable statistics across strata
4. Noise and redundancy risk assessment
5. Chunking hypotheses per stratum
6. JS deep-dive: sub-corpus filtering, 3 chunking strategies compared
7. Phased implementation plan
8. Summary and open questions

In [ ]:
import json
import os
import re
from pathlib import Path
from collections import Counter, defaultdict

BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / "data" / "kdk").exists():
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data" / "kdk"
print(f"Data directory: {DATA_DIR.resolve()}")

In [ ]:
# ── Shared helpers (same as NB01) ──
EXCLUDED_DIRS = {
    "node_modules", ".git", "dist", "build", ".output",
    "coverage", ".nyc_output", ".c8", "__pycache__", ".cache", ".github",
}
EXCLUDED_FILES = {"yarn.lock", "CHANGELOG.md"}
EXCLUDED_EXTENSIONS = {".png", ".gif", ".tif", ".svg", ".lock"}

def scan_files(root, excluded_dirs):
    files = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in excluded_dirs]
        for f in filenames:
            files.append(Path(dirpath) / f)
    return files

def fmt_size(nbytes):
    for unit in ["B", "KB", "MB", "GB"]:
        if nbytes < 1024:
            return f"{nbytes:.1f} {unit}"
        nbytes /= 1024
    return f"{nbytes:.1f} TB"

def read_text(path):
    return path.read_text(encoding="utf-8", errors="ignore")

def estimate_tokens(text):
    return len(text) // 4

all_files = [
    f for f in scan_files(DATA_DIR, EXCLUDED_DIRS)
    if f.name not in EXCLUDED_FILES and f.suffix.lower() not in EXCLUDED_EXTENSIONS
]
print(f"Retained files: {len(all_files)}")

---
## 1. Official corpus stratification

We define 16 strata that will serve as the stable reference for all chunking experiments.

In [ ]:
def classify_stratum(path):
    rel = path.relative_to(DATA_DIR)
    parts = rel.parts
    rel_str = str(rel).lower()
    ext = path.suffix.lower()
    name = path.name.lower()

    if parts[0] == "docs" and ext == ".md":
        return "documentation_markdown"
    if ext == ".vue":
        return "vue_sfc"
    if path.name == "index.js":
        return "barrel_index_js"
    if ext == ".json":
        if name == "package.json":
            return "json_package_tooling"
        if "schemas" in parts or name.endswith("create.json") or name.endswith("update.json"):
            return "json_schemas"
        if "/i18n/" in rel_str or name.endswith("_fr.json") or name.endswith("_en.json"):
            return "json_i18n"
        if parts[0] == "test":
            return "json_test_data"
        if parts[0] == "docs":
            return "json_docs_meta"
        return "json_other"
    if parts[0] == "test":
        return "tests"
    if ext in {".js", ".mjs", ".cjs"}:
        return "code_js"
    if ext in {".html", ".ejs"}:
        return "templates"
    if ext in {".md", ".txt"}:
        return "text_outside_docs"
    if ext in {".xml", ".drawio"}:
        return "diagrams_and_specs"
    if ext in {".yml", ".yaml", ".toml", ".properties"}:
        return "config_non_json"
    return "other"

by_stratum = defaultdict(list)
for f in all_files:
    by_stratum[classify_stratum(f)].append(f)

print(f"{'Stratum':<24} {'Files':>8} {'Size':>12}")
print("-" * 48)
for stratum, files in sorted(by_stratum.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"{stratum:<24} {len(files):>8} {fmt_size(sum(f.stat().st_size for f in files)):>12}")

---
## 2. Decision matrix

For each stratum: index type (primary / secondary / metadata-only / excluded),
role in the RAG system, and implementation priority.

In [ ]:
STRATUM_STRATEGY = {
    "documentation_markdown": {"index": "primary",   "role": "main recall",         "chunk": "Markdown sections by heading",    "priority": "phase_1"},
    "code_js":               {"index": "primary",   "role": "main recall",         "chunk": "export / function / class",      "priority": "phase_1"},
    "vue_sfc":               {"index": "primary",   "role": "main recall",         "chunk": "per component (script focus)",   "priority": "phase_1"},
    "json_schemas":          {"index": "primary",   "role": "main recall",         "chunk": "whole schema or property group", "priority": "phase_1"},
    "barrel_index_js":       {"index": "metadata",  "role": "navigation context",  "chunk": "re-export map only",             "priority": "phase_2"},
    "json_i18n":             {"index": "secondary", "role": "lexical support",      "chunk": "by key prefix or whole file",   "priority": "phase_2"},
    "config_non_json":       {"index": "secondary", "role": "config context",       "chunk": "whole file",                    "priority": "phase_2"},
    "json_other":            {"index": "evaluate",  "role": "to be qualified",      "chunk": "TBD",                           "priority": "phase_2"},
    "diagrams_and_specs":    {"index": "secondary", "role": "comprehension aid",    "chunk": "whole file if text",             "priority": "phase_3"},
    "json_docs_meta":        {"index": "secondary", "role": "doc support",          "chunk": "whole file",                    "priority": "phase_3"},
    "templates":             {"index": "secondary", "role": "occasional context",   "chunk": "whole file",                    "priority": "phase_3"},
    "tests":                 {"index": "excluded",  "role": "debug support",        "chunk": "per test block",                "priority": "phase_3"},
    "text_outside_docs":     {"index": "secondary", "role": "supplementary",        "chunk": "whole file",                    "priority": "phase_3"},
    "json_test_data":        {"index": "excluded",  "role": "out of scope",         "chunk": "none",                          "priority": "pause"},
    "json_package_tooling":  {"index": "excluded",  "role": "tooling noise",        "chunk": "none",                          "priority": "pause"},
    "other":                 {"index": "evaluate",  "role": "unclassified",         "chunk": "TBD",                           "priority": "pause"},
}

print(f"{'Stratum':<24} {'Index':<12} {'Role':<22} {'Priority'}")
print("-" * 76)
for stratum in sorted(by_stratum):
    s = STRATUM_STRATEGY.get(stratum, {"index": "TBD", "role": "TBD", "priority": "TBD"})
    print(f"{stratum:<24} {s['index']:<12} {s['role']:<22} {s['priority']}")

---
## 3. Comparable statistics across strata

In [ ]:
def semantic_density(path, text):
    ext = path.suffix.lower()
    if ext in {".js", ".mjs", ".cjs"}:
        exports = len(re.findall(r"^export\s", text, re.MULTILINE))
        functions = len(re.findall(r"^(?:export\s+)?(?:async\s+)?function\s+", text, re.MULTILINE))
        classes = len(re.findall(r"^(?:export\s+)?class\s+", text, re.MULTILINE))
        return exports + functions + classes
    if ext == ".vue":
        return len(re.findall(r"\bprops\b", text)) + len(re.findall(r"\bemits\b", text)) + len(re.findall(r"\buse[A-Z]\w+", text))
    if ext == ".md":
        h2 = len(re.findall(r"^##\s+", text, re.MULTILINE))
        h3 = len(re.findall(r"^###\s+", text, re.MULTILINE))
        code = len(re.findall(r"^```", text, re.MULTILINE)) // 2
        return h2 + h3 + code
    if ext == ".json":
        try:
            obj = json.loads(text)
            return len(obj) if isinstance(obj, (dict, list)) else 1
        except Exception:
            return 0
    return 0

stratum_stats = []
stratum_details = {}
for stratum, files in by_stratum.items():
    token_list = []
    details = []
    for f in files:
        text = read_text(f)
        tokens = estimate_tokens(text)
        score = semantic_density(f, text)
        token_list.append(tokens)
        details.append({"file": str(f.relative_to(DATA_DIR)), "tokens": tokens, "score": score})
    details.sort(key=lambda x: x["tokens"], reverse=True)
    stratum_details[stratum] = details
    sorted_tokens = sorted(token_list)
    median = sorted_tokens[len(sorted_tokens) // 2] if sorted_tokens else 0
    stratum_stats.append({
        "stratum": stratum, "files": len(files), "tokens": sum(token_list),
        "median": median, "max": max(token_list) if token_list else 0,
        "max_score": max((d["score"] for d in details), default=0),
    })

print(f"{'Stratum':<24} {'Files':>6} {'~Tokens':>10} {'Median':>8} {'Max':>8} {'Density':>8}")
print("-" * 70)
for row in sorted(stratum_stats, key=lambda x: x["tokens"], reverse=True):
    print(f"{row['stratum']:<24} {row['files']:>6} {row['tokens']:>10} {row['median']:>8} {row['max']:>8} {row['max_score']:>8}")

In [ ]:
# Top 5 files per stratum
print("Largest files per stratum:")
for stratum in sorted(stratum_details):
    print(f"\n[{stratum}]")
    for item in stratum_details[stratum][:5]:
        print(f"  {item['file']:<60} {item['tokens']:>8} tok  density={item['score']}")

---
## 4. Noise and redundancy risks

In [ ]:
NOISE_RISKS = {
    "barrel_index_js":      "High redundancy with actual exported modules",
    "json_test_data":       "Very high token cost, low recall value",
    "json_package_tooling": "Tooling and build noise",
    "tests":                "Useful behaviors but secondary for first iteration",
    "templates":            "Short, low-discrimination content",
}

print(f"{'Stratum':<24} {'Risk assessment'}")
print("-" * 80)
for stratum, risk in NOISE_RISKS.items():
    if stratum in by_stratum:
        print(f"{stratum:<24} {risk}")

---
## 5. Chunking hypotheses per stratum

These are implementation-ready candidates, not just intuitions.

In [ ]:
CHUNKING_HYPOTHESES = {
    "documentation_markdown": [
        "Base: split at `##`",
        "Re-split at `###` if chunk exceeds ~1500 tokens",
        "Merge adjacent sections if chunk is below ~50 tokens",
    ],
    "code_js": [
        "Base: one chunk per `export function`, `export class`, or major exported symbol",
        "Fallback: group nearby top-level functions when no explicit exports",
        "Attach local helpers to the nearest exported symbol",
    ],
    "vue_sfc": [
        "Variant A: one chunk per `.vue` component",
        "Variant B: separate `script` chunk + structural `template` summary",
        "Preserve component metadata (props, emits, imported composables)",
    ],
    "json_schemas": [
        "Variant A: one chunk per complete schema",
        "Variant B: one chunk per root property group if schema is large",
        "Mandatory metadata: $id, title, required, property names",
    ],
    "barrel_index_js": [
        "No primary chunks — extract re-export map as navigation metadata only",
    ],
    "json_i18n": [
        "Chunk by key prefix or whole file depending on size",
        "Use as lexical support, not primary recall",
    ],
    "tests": [
        "Exclude initially, or build a separate index",
        "If kept: one chunk per file or per describe/it block",
    ],
}

for stratum in sorted(CHUNKING_HYPOTHESES):
    print(f"[{stratum}]")
    for h in CHUNKING_HYPOTHESES[stratum]:
        print(f"  - {h}")
    print()

---
## 6. JS deep-dive: chunking strategy comparison

The `code_js` stratum is the largest primary stratum. We compare three approaches:
- **whole_file**: one chunk per file
- **top_level_units**: one chunk per detected symbol
- **exports_first**: one chunk per exported symbol + residual helper chunk

In [ ]:
# ── 6a. Sub-corpus filtering ──
code_candidates = [f for f in all_files if f.suffix.lower() in {".js", ".mjs", ".cjs"}]

def js_sub_stratum(path):
    rel = path.relative_to(DATA_DIR)
    rel_str = str(rel).lower()
    if rel.parts[0] == "test" or "tests" in rel.parts:
        return "tests_js"
    if path.name.lower() == "index.js":
        return "barrel_index_js"
    if path.name.endswith(".min.js") or "/libs/" in rel_str:
        return "vendor_minified"
    return "code_js"

js_substrata = defaultdict(list)
for f in code_candidates:
    js_substrata[js_sub_stratum(f)].append(f)

print(f"{'Sub-stratum':<20} {'Files':>6} {'~Tokens':>10}")
print("-" * 40)
for sub, files in sorted(js_substrata.items(), key=lambda x: len(x[1]), reverse=True):
    tokens = sum(estimate_tokens(read_text(f)) for f in files)
    print(f"{sub:<20} {len(files):>6} {tokens:>10}")

code_js_files = sorted(js_substrata["code_js"])
print(f"\nRetained for analysis: {len(code_js_files)} files")

In [ ]:
# ── 6b. Semantic profile ──
profiles = []
for f in code_js_files:
    text = read_text(f)
    rel = f.relative_to(DATA_DIR)
    exports = len(re.findall(r"^export\s", text, re.MULTILINE))
    functions = len(re.findall(r"^(?:export\s+)?(?:async\s+)?function\s+", text, re.MULTILINE))
    classes = len(re.findall(r"^(?:export\s+)?class\s+", text, re.MULTILINE))
    consts = len(re.findall(r"^(?:export\s+)?const\s+", text, re.MULTILINE))
    tokens = estimate_tokens(text)
    profiles.append({"file": str(rel), "zone": rel.parts[0] if len(rel.parts) > 1 else "(root)",
                     "tokens": tokens, "exports": exports, "functions": functions,
                     "classes": classes, "consts": consts, "score": exports + functions + classes + consts})

print("Top 15 semantically densest JS files:")
for p in sorted(profiles, key=lambda x: x["score"], reverse=True)[:15]:
    print(f"  {p['file']:<50} score={p['score']:>3} exp={p['exports']:>2} fn={p['functions']:>2} cls={p['classes']:>2} const={p['consts']:>2}")

In [ ]:
# ── 6c. Heuristic top-level unit extraction ──
TOP_LEVEL_PATTERNS = [
    ("export_function", re.compile(r"^export\s+(?:async\s+)?function\s+([A-Za-z_$][\w$]*)", re.MULTILINE)),
    ("export_class",    re.compile(r"^export\s+class\s+([A-Za-z_$][\w$]*)", re.MULTILINE)),
    ("export_const",    re.compile(r"^export\s+const\s+([A-Za-z_$][\w$]*)", re.MULTILINE)),
    ("export_let",      re.compile(r"^export\s+let\s+([A-Za-z_$][\w$]*)", re.MULTILINE)),
    ("function",        re.compile(r"^(?:async\s+)?function\s+([A-Za-z_$][\w$]*)", re.MULTILINE)),
    ("class",           re.compile(r"^class\s+([A-Za-z_$][\w$]*)", re.MULTILINE)),
    ("const",           re.compile(r"^const\s+([A-Za-z_$][\w$]*)", re.MULTILINE)),
    ("let",             re.compile(r"^let\s+([A-Za-z_$][\w$]*)", re.MULTILINE)),
]

def extract_top_level_units(path):
    text = read_text(path)
    units = []
    for unit_type, pattern in TOP_LEVEL_PATTERNS:
        for m in pattern.finditer(text):
            line = text.count("\n", 0, m.start()) + 1
            units.append({"type": unit_type, "name": m.group(1), "start": m.start(), "line": line})
    units.sort(key=lambda u: u["start"])
    for i, u in enumerate(units):
        end = units[i + 1]["start"] if i + 1 < len(units) else len(text)
        u["end"] = end
        u["tokens"] = estimate_tokens(text[u["start"]:end])
    return units

In [ ]:
# ── 6d. Three strategies compared ──
def chunks_whole_file(path, label="whole_file"):
    return [{"strategy": label, "file": str(path.relative_to(DATA_DIR)),
             "name": path.stem, "tokens": estimate_tokens(read_text(path))}]

def chunks_top_level(path):
    units = extract_top_level_units(path)
    if not units:
        return chunks_whole_file(path, "top_level_units")
    return [{"strategy": "top_level_units", "file": str(path.relative_to(DATA_DIR)),
             "type": u["type"], "name": u["name"], "tokens": u["tokens"]} for u in units]

def chunks_exports_first(path):
    text = read_text(path)
    units = extract_top_level_units(path)
    exports = [u for u in units if u["type"].startswith("export_")]
    if not exports:
        return chunks_whole_file(path, "exports_first")
    chunks = [{"strategy": "exports_first", "file": str(path.relative_to(DATA_DIR)),
               "type": u["type"], "name": u["name"], "tokens": u["tokens"]} for u in exports]
    # Residual non-export code
    export_ranges = [(u["start"], u["end"]) for u in exports]
    residual_parts = []
    cursor = 0
    for start, end in export_ranges:
        if start > cursor:
            residual_parts.append(text[cursor:start])
        cursor = max(cursor, end)
    if cursor < len(text):
        residual_parts.append(text[cursor:])
    residual = "".join(residual_parts).strip()
    if residual and estimate_tokens(residual) > 20:
        chunks.append({"strategy": "exports_first", "file": str(path.relative_to(DATA_DIR)),
                       "type": "residual", "name": "__helpers__", "tokens": estimate_tokens(residual)})
    return chunks

STRATEGIES = {
    "whole_file": chunks_whole_file,
    "top_level_units": chunks_top_level,
    "exports_first": chunks_exports_first,
}

all_sim_chunks = []
for f in code_js_files:
    for name, fn in STRATEGIES.items():
        all_sim_chunks.extend(fn(f))

print(f"{'Strategy':<20} {'Chunks':>8} {'Median':>8} {'Mean':>8} {'<50 tok':>8} {'>1500 tok':>10}")
print("-" * 66)
for strategy in STRATEGIES:
    chunks = [c for c in all_sim_chunks if c["strategy"] == strategy]
    tokens = sorted(c["tokens"] for c in chunks)
    small = sum(1 for t in tokens if t < 50)
    large = sum(1 for t in tokens if t > 1500)
    mean = sum(tokens) // len(tokens)
    median = tokens[len(tokens) // 2]
    print(f"{strategy:<20} {len(chunks):>8} {median:>8} {mean:>8} {small:>8} {large:>10}")

In [ ]:
# ── 6e. Interpretation ──
print("Strategy assessment:")
print("  - whole_file:       Too coarse — good local coherence but too many oversized chunks")
print("  - top_level_units:  Too fine alone — many small chunks with little standalone information")
print("  - exports_first:    Best trade-off — still needs helper-attachment heuristic")
print()
print("Recommendation: use exports_first as base strategy for code_js")
print("Open question: how to attach non-exported helpers to their nearest export without full AST parsing")

---
## 7. Phased implementation plan

In [ ]:
phases = defaultdict(list)
for stratum in sorted(by_stratum):
    priority = STRATUM_STRATEGY.get(stratum, {}).get("priority", "TBD")
    phases[priority].append(stratum)

for phase in ["phase_1", "phase_2", "phase_3", "pause"]:
    if phase not in phases:
        continue
    print(f"[{phase}]")
    for s in phases[phase]:
        strategy = STRATUM_STRATEGY[s]
        print(f"  {s:<24} -> {strategy['chunk']}")
    print()

---
## 8. Summary and open questions

### Key decisions

| Decision | Rationale |
|---|---|
| 16-stratum classification | Stable reference for all experiments |
| Phase 1: `documentation_markdown`, `code_js`, `vue_sfc`, `json_schemas` | Core content, highest recall value |
| Exclude test data and tooling initially | ~900k tokens saved, minimal recall loss |
| `exports_first` for JS | Best balance between precision and chunk coherence |
| Markdown split at `##` | Matches document structure; re-split at `###` for outliers |
| Barrel files as metadata only | High redundancy with actual modules |

### Open questions

1. How to attach non-exported JS helpers to their nearest exported symbol without full AST?
2. Vue SFC: whole component vs script/template separation — needs A/B testing
3. JSON schemas: whole schema vs property groups for large schemas
4. Should i18n translations be in the primary index or a secondary lookup?

In [ ]:
phase_1 = phases.get("phase_1", [])
phase_2 = phases.get("phase_2", [])
phase_3 = phases.get("phase_3", [])
paused = phases.get("pause", [])

print("=" * 72)
print("SUMMARY — Stratification & Chunking Strategy")
print("=" * 72)
print(f"""
CORPUS:
  Retained files: {len(all_files)}
  Strata identified: {len(by_stratum)}

IMPLEMENTATION PHASES:
  Phase 1 (primary index): {', '.join(phase_1)}
  Phase 2 (secondary):     {', '.join(phase_2) if phase_2 else 'none'}
  Phase 3 (deferred):      {', '.join(phase_3) if phase_3 else 'none'}
  Paused:                  {', '.join(paused) if paused else 'none'}

JS CHUNKING STRATEGY:
  Recommended: exports_first
  Sub-corpus: {len(code_js_files)} files (excl. vendor, barrel, test)

NEXT: NB03 (comparison notebook) applies these strategies in a live
RAG pipeline evaluation against LangChain defaults.
""")